# Cotizador de compras con LangGraph y LangChain

**Entregable 4 — Técnicas Avanzadas de IA**  
Camilo Andrés Armenta · Simón Sánchez Gómez

Este cuaderno recorre el sistema completo, pieza por pieza, hasta ejecutarlo de
extremo a extremo. No redefine el código: lo **importa** del paquete `cotizador`
y muestra el fuente real con `inspect`, así lo que se lee aquí es exactamente lo
que corre en producción.

## El flujo

```
solicitud en lenguaje natural
        |
  interpretar_pedido      (LLM)      texto  ->  Pedido estructurado
        |
  preparar_busqueda       (código)   ¿en qué tiendas se busca?
        |
  agente_buscador    <->  herramientas        ciclo: el agente decide, las
        |                 (código)            herramientas buscan de verdad
  recolectar_ofertas      (código)   agrupa las ofertas por producto
        |
  normalizar_ofertas      (código)   precio por kg, L o unidad
        |
  clasificar_equivalencia (LLM)      equivalente / sustituto / no comparable
        |
  calcular_presupuesto    (código)   por tienda y compra combinada
```

**Criterio de diseño:** el LLM solo interviene donde hay lenguaje. Las cuentas van
en código para que sean reproducibles y se puedan comprobar a mano.

## 0. Preparación

Requiere `OPENAI_API_KEY` en el archivo `.env` de la raíz del proyecto.
Las celdas marcadas **(sin clave)** funcionan sin ella.

In [ ]:
from pathlib import Path
import sys

# El cuaderno vive en notebooks/: se trabaja desde la raíz del proyecto.
RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
import os

load_dotenv(RAIZ / '.env')
clave = os.getenv('OPENAI_API_KEY', '')
print('Carpeta de trabajo :', RAIZ.name)
print('Clave cargada      :', 'sí' if clave and not clave.startswith('sk-...') else 'NO')
print('Modelo             :', os.getenv('OPENAI_MODEL', 'gpt-4o-mini'))

In [ ]:
import inspect

def ver(objeto):
    """Muestra el código fuente real de una clase o función del proyecto."""
    print(inspect.getsource(objeto))

## 1. Los modelos de datos (Pydantic) — *(sin clave)*

Un esquema Pydantic es un contrato: define qué campos existen y qué valores se
aceptan. Si un dato no cumple, se rechaza en vez de pasar desapercibido.

Hay tres familias:

| Modelo | Qué es | Quién lo llena |
|---|---|---|
| `Pedido` / `ItemPedido` | Lo que el LLM debe devolver al interpretar | El LLM |
| `Oferta` | Un precio publicado por una tienda | Los catálogos |
| `EstadoCompra` | La memoria compartida del grafo | Los nodos |

In [ ]:
from cotizador.schemas import EstadoCompra, ItemPedido, Oferta, Pedido

ver(ItemPedido)

In [ ]:
# Un pedido de ejemplo, armado a mano para ver la forma de los datos.
ejemplo = Pedido(
    items=[
        ItemPedido(texto_original='2 kg de arroz', producto='arroz', cantidad=2,
                   unidad='kg', marca=None, presentacion=None, observaciones=None),
    ],
    ciudad='Medellín',
    proveedores=['Éxito', 'Carulla'],
)
print(ejemplo.model_dump_json(indent=2))

In [ ]:
# El contrato se hace valer: una unidad que el sistema no sabe convertir se rechaza.
from pydantic import ValidationError

try:
    ItemPedido(texto_original='2 arrobas de papa', producto='papa', cantidad=2,
               unidad='arroba', marca=None, presentacion=None, observaciones=None)
except ValidationError as error:
    print('Rechazado, como debe ser:')
    print(error.errors()[0]['msg'])

## 2. Los catálogos — *(sin clave)*

Precios reales tomados de las fuentes públicas de cada tienda el 11 de noviembre
de 2026. Cada fila conserva su URL y su fecha: eso es lo que el enunciado llama
*fuentes verificables*.

Éxito y Carulla publican el precio en la página de cada producto. Ara no tiene
tienda en línea con precio por producto: sus datos salen del folleto semanal.

In [ ]:
from cotizador.catalogos import cargar_catalogos

catalogos = cargar_catalogos()
for proveedor, ofertas in catalogos.items():
    print(f'{proveedor:10} {len(ofertas):3} ofertas')

In [ ]:
# Cómo se ve una oferta por dentro.
print(catalogos['Éxito'][0].model_dump_json(indent=2))

## 3. Búsqueda y herramientas — *(sin clave)*

Una **herramienta** es una función que el LLM puede decidir llamar. Él no la
ejecuta: devuelve una *llamada a herramienta* con los argumentos, el programa la
ejecuta y le entrega el resultado.

La búsqueda compara **palabras normalizadas**, no subcadenas. Por eso "papa" no
trae "papel higiénico" y "huevos" sí encuentra "Huevos ... (30 und)".

In [ ]:
from cotizador.busqueda import buscar_ofertas, palabras_clave

print('Azúcares  ->', palabras_clave('Azúcares'))
print('papel higiénico ->', palabras_clave('papel higiénico'))
print()
print('Buscando "papa" en Éxito:')
for oferta in buscar_ofertas(catalogos['Éxito'], 'papa'):
    print('  ', oferta.nombre)
print('(no aparece papel higiénico: se comparan palabras, no subcadenas)')

In [ ]:
from cotizador.herramientas import crear_herramientas

herramientas = crear_herramientas(catalogos)
print('Herramientas disponibles:', [h.name for h in herramientas])
print()
print('Esto es lo que el modelo lee para decidir cuándo usarla:')
print(herramientas[-1].description)
print('Argumentos:', list(herramientas[-1].args_schema.model_json_schema()['properties']))

In [ ]:
# Se invoca igual que lo haría el grafo, y devuelve DOS cosas.
mensaje = herramientas[-1].invoke(
    {'name': herramientas[-1].name, 'args': {'termino': 'arroz'},
     'id': 'demo-1', 'type': 'tool_call'}
)
print('--- texto que lee el modelo ---')
print(mensaje.content[:400])
print()
print('--- objetos Oferta que usa el código ---')
print(len(mensaje.artifact), 'ofertas, la primera:', mensaje.artifact[0].nombre)
print('con su URL:', mensaje.artifact[0].url)

## 4. Unidades y precio unitario — *(sin clave)*

Dos ofertas del mismo producto no se comparan por el precio de la etiqueta si
vienen en empaques distintos. Hay que llevarlas a una unidad base.

Las conversiones se guardan como **fracción** (numerador y denominador) para que
1000 g den exactamente 1 kg: un error de coma flotante, al redondear paquetes,
haría comprar uno de más.

In [ ]:
from cotizador.unidades import a_base, convertir, paquetes_necesarios, precio_por_unidad

print('1000 g en kg        :', a_base(1000, 'g'))
print('1 libra en g        :', convertir(1, 'lb', 'g'), '(libra de mercado colombiana)')
print()
print('Arroz 5 kg a $21.950:', round(precio_por_unidad(21950, 5000, 'g')), 'por kg')
print('Arroz 1 kg a $6.790 :', round(precio_por_unidad(6790, 1000, 'g')), 'por kg')
print()
print('Para 5 kg con bolsas de 3 kg hacen falta', paquetes_necesarios(5, 'kg', 3000, 'g'), 'bolsas')

In [ ]:
# El precio de la etiqueta engaña: comparación con datos reales del catálogo.
from cotizador.nodos.normalizar import normalizar_ofertas
from cotizador.schemas import ResultadoBusqueda

item = ItemPedido(texto_original='2 kg de arroz', producto='arroz', cantidad=2,
                  unidad='kg', marca=None, presentacion=None, observaciones=None)
ofertas = [o for p in ('Éxito', 'Carulla') for o in buscar_ofertas(catalogos[p], 'arroz')]
estado = EstadoCompra(resultados=[ResultadoBusqueda(item=item, ofertas=ofertas)])

for n in normalizar_ofertas(estado)['resultados'][0].normalizadas[:5]:
    etiqueta = f'$ {n.oferta.precio:,.0f}'.replace(',', '.')
    unitario = f'$ {n.precio_unitario:,.0f}'.replace(',', '.')
    print(f'{n.oferta.proveedor:9} {n.oferta.nombre[:40]:42} {etiqueta:>9}  ->  {unitario:>8} por kg')

## 5. La cadena de interpretación (LLM)

Una **cadena** de LangChain es una tubería: la salida de cada pieza es la entrada
de la siguiente.

```
{'texto_usuario': '...'}  ->  prompt  ->  LLM con salida estructurada  ->  Pedido
```

`with_structured_output(..., strict=True)` obliga al modelo a responder un JSON
que cumpla el esquema de `Pedido`. No puede devolver texto libre.

In [ ]:
from cotizador.prompts import crear_prompt_interprete

# Lo que recibe el modelo: dos mensajes. El del usuario va delimitado y
# declarado como DATO (defensa contra inyección de instrucciones).
sistema, humano = crear_prompt_interprete().format_messages(
    texto_usuario='Necesito 2 kg de arroz y 30 huevos para Medellín'
)
print(f'--- system ({len(sistema.content)} caracteres) ---')
print(sistema.content[:700], '...')
print()
print('--- human ---')
print(humano.content)

In [ ]:
# A partir de aquí se consume la clave de OpenAI.
from cotizador.llm import construir_interprete

pedido = construir_interprete().invoke(
    {'texto_usuario': 'Necesito 2 kg de arroz integral, 1 litro de aceite de girasol '
                      'y 30 huevos. Consulta Exito y Carulla.'}
)
print(pedido.model_dump_json(indent=2))

## 6. Los nodos y el estado — *(sin clave)*

Un **nodo** de LangGraph es una función normal: recibe el estado y devuelve un
diccionario con **solo los campos que cambia**. No hay que confundirlo con la
cadena:

| | Cadena | Nodo |
|---|---|---|
| Qué es | Tubería de LangChain | Función de Python |
| Recibe | `{'texto_usuario': ...}` | El `EstadoCompra` completo |
| Devuelve | Un `Pedido` | `{'pedido': ..., 'historial': [...]}` |

El nodo es el puente entre el grafo y la cadena.

In [ ]:
from cotizador.nodos import crear_nodo_interpretar

ver(crear_nodo_interpretar)

In [ ]:
# El nodo se prueba con una cadena falsa: sin red y sin gastar clave.
from langchain_core.runnables import RunnableLambda

cadena_falsa = RunnableLambda(lambda entrada: ejemplo)
nodo = crear_nodo_interpretar(cadena_falsa)
salida = nodo(EstadoCompra(texto_usuario='2 kg de arroz'))

print('Campos que actualiza:', list(salida))
print('Historial          :', salida['historial'][0])

### El ciclo del agente buscador

Es el único nodo con ciclo, y el corazón del sistema. Tiene cuatro protecciones:

1. **El encargo lo arma el código**, no el modelo: así no se omite ningún producto.
2. **Tope de vueltas**: si el agente insiste, el router corta el ciclo.
3. **Red de seguridad**: si un producto queda sin ofertas, el código busca por su cuenta.
4. **El modelo no toca los precios**: viajan como objetos, no como texto copiado.

In [ ]:
from cotizador.routers import enrutar_busqueda
from cotizador.nodos.buscar import MAX_VUELTAS

ver(enrutar_busqueda)
print('Tope de vueltas del ciclo:', MAX_VUELTAS)

## 7. El grafo completo — *(sin clave para dibujarlo)*

`StateGraph` recibe la forma del estado, los nodos y las aristas. Las aristas
**condicionales** son las que deciden el camino en tiempo de ejecución.

In [ ]:
from cotizador.grafo import construir_grafo

ver(construir_grafo)

In [ ]:
# Para dibujar no hace falta la clave: se inyectan piezas de relleno.
from IPython.display import Image, display

relleno = RunnableLambda(lambda _: None)
grafo_dibujo = construir_grafo(interprete=relleno, buscador=relleno, clasificador=relleno)

try:
    display(Image(grafo_dibujo.get_graph().draw_mermaid_png()))
except Exception as error:
    print('No se pudo generar la imagen:', type(error).__name__)
    print('Pega este código en https://mermaid.live')
    print(grafo_dibujo.get_graph().draw_mermaid())

## 8. Ejecución de extremo a extremo

Ahora sí, el sistema completo con el modelo real. Esta celda consume clave:
una llamada para interpretar, una o dos para el agente buscador y una por
producto para clasificar.

In [ ]:
solicitud = ('Necesito 2 kg de arroz, 1 litro de aceite de girasol y 30 huevos. '
             'Consulta Exito y Carulla.')

grafo = construir_grafo()
final = EstadoCompra.model_validate(grafo.invoke(EstadoCompra(texto_usuario=solicitud)))

print('--- Traza de nodos ---')
for linea in final.historial:
    print(linea)

In [ ]:
# Lo que el agente pidió y lo que las herramientas respondieron.
from langchain_core.messages import AIMessage, ToolMessage

for mensaje in final.mensajes:
    if isinstance(mensaje, AIMessage) and mensaje.tool_calls:
        for llamada in mensaje.tool_calls:
            print(f"  el agente pide  -> {llamada['name']}(termino={llamada['args']['termino']!r})")
    elif isinstance(mensaje, ToolMessage):
        print(f'  la herramienta responde <- {len(mensaje.artifact or [])} ofertas')
    elif isinstance(mensaje, AIMessage) and mensaje.content:
        print('  el agente cierra:', mensaje.content[:90])

## 9. Las ofertas clasificadas

`=` equivalente · `~` sustituto · `x` no comparable

In [ ]:
marca = {'equivalente': '=', 'sustituto': '~', 'no_comparable': 'x'}

for resultado in final.resultados:
    print(f'--- {resultado.item.producto} ({len(resultado.normalizadas)} ofertas) ---')
    for n in resultado.normalizadas[:4]:
        precio = f'$ {n.oferta.precio:,.0f}'.replace(',', '.')
        unitario = (f'$ {n.precio_unitario:,.0f}'.replace(',', '.') + f' por {n.unidad_base}'
                    if n.comparable else 'no comparable')
        print(f'  {marca.get(n.equivalencia, " ")} [{n.oferta.proveedor}] '
              f'{n.oferta.nombre[:40]:42} {precio:>9}  {unitario}')
        if n.razon:
            print(f'      {n.razon}')
    print()

## 10. El presupuesto

Tres reglas que se ven en el resultado:

- **No se compra fraccionado**: se informa el sobrante.
- **Se minimiza el costo de lo pedido**, no el precio por kilo.
- **Un envío desconocido no es cero**: sin él no hay total confirmado.

In [ ]:
def pesos(valor):
    return '$ ' + f'{valor:,.0f}'.replace(',', '.')

def mostrar(presupuesto):
    print(f'--- {presupuesto.nombre} ---')
    for linea in presupuesto.lineas:
        if not linea.cubierto:
            print(f'  {linea.producto}: SIN COTIZACIÓN ({linea.motivo})')
            continue
        tienda = f' [{linea.oferta.proveedor}]' if presupuesto.combinado else ''
        sustituto = '  (sustituto)' if linea.equivalencia == 'sustituto' else ''
        sobra = (f', sobran {linea.sobrante:g} {linea.unidad_base}'
                 if linea.sobrante and linea.sobrante > 0 else '')
        print(f'  {linea.producto}{tienda}: {linea.paquetes} x {linea.oferta.nombre}')
        print(f'      {pesos(linea.subtotal)}{sobra}{sustituto}')
    print(f'  Subtotal productos : {pesos(presupuesto.subtotal_productos)}')
    print('  Envío              : no verificado')
    print('  Total confirmado   : no se puede calcular sin el envío')
    if presupuesto.sin_cotizacion:
        print(f'  Sin cotización     : {", ".join(presupuesto.sin_cotizacion)}')
    print()

for presupuesto in final.presupuestos:
    mostrar(presupuesto)
if final.combinado:
    mostrar(final.combinado)
else:
    print('La compra combinada no aporta frente a comprar todo en una sola tienda.')

## 11. Las pruebas automáticas — *(sin clave)*

Todo el sistema se prueba sin red y sin consumir clave: el LLM se reemplaza por
funciones falsas con guion. Así se verifica la **orquestación** (quién llama a
quién, cuándo se corta el ciclo, qué se recolecta) sin depender de lo que decida
un modelo real.

In [ ]:
import subprocess

resultado = subprocess.run(['uv', 'run', 'pytest', '-q'], cwd=RAIZ,
                           capture_output=True, text=True)
print(resultado.stdout[-800:] or resultado.stderr[-800:])

## Cierre

El sistema cubre lo que pide el enunciado:

| Requisito | Dónde se resuelve |
|---|---|
| Identificar producto, cantidad, unidad, restricciones, ciudad y proveedores | `interpretar_pedido` |
| Consultar al menos dos proveedores | `agente_buscador` + herramientas |
| Conservar nombre, precio, moneda, URL, fecha y disponibilidad | modelo `Oferta` |
| Convertir unidades y comparar por precio unitario | `normalizar_ofertas` |
| Distinguir equivalente, sustituto y no comparable | `clasificar_equivalencia` |
| Presupuesto por tienda y alternativa combinada | `calcular_presupuesto` |
| Envío desconocido no se asume cero | `Presupuesto.total_confirmado` |

Los detalles de cada decisión, los fallos encontrados y sus correcciones están en
`Memoria del Proyecto.docx` y en `Avances por actividad/`.